# NB03 · Semana 7 — DuckDB y Delta Lake sobre el lake

**Ingeniería de Datos — LCD · UCA Rosario · 2026 · grupo07**

## Objetivo

Consultar el bronze (Delta Lake en MinIO) con **DuckDB**, explorarlo para encontrar los problemas que tiene que limpiar silver, y ver el **historial de versiones** de Delta (*time travel*).

Cada hallazgo de la sección 2 termina en una decisión de `stg_urlhaus` (ver `docs/entregas/E2_dbt.md`).

In [1]:
# === Configuración del entorno — NO tocar ===
import os, sys
_d = os.getcwd()
while True:
    if os.path.exists(os.path.join(_d, "entorno.py")):
        sys.path.insert(0, _d); break
    _p = os.path.dirname(_d)
    if _p == _d:
        raise FileNotFoundError("No encuentro entorno.py subiendo desde el directorio actual.")
    _d = _p
from entorno import BUCKET, BRONZE, STORAGE_OPTIONS, info
info()

Entorno detectado: servidor de la cátedra  (grupo: grupo07)
  Lake / bucket   : grupo07   →   bronze en s3://grupo07/bronze/series
  PostgreSQL      : base 'warehouse', usuario 'grupo07', host 'postgres'
  Material en     : /home/jovyan/work


In [2]:
# Dependencias: el JupyterHub puede reiniciarse y perder lo instalado con pip
%pip install -q -r {os.path.join(_d, 'requirements.txt')}

Note: you may need to restart the kernel to use updated packages.


## 1. Leer el bronze

`deltalake` lee la versión vigente de la tabla (solo los archivos que la componen) y DuckDB la consulta con SQL, sin cargarla en ninguna base.

In [3]:
import duckdb
import pandas as pd
from deltalake import DeltaTable

dt = DeltaTable(BRONZE, storage_options=STORAGE_OPTIONS)
bronze = dt.to_pyarrow_table()
con = duckdb.connect()
con.sql("set TimeZone = 'UTC'")          # las fechas de URLhaus están en UTC
con.register("bronze", bronze)
pd.set_option("display.max_colwidth", 80)

print(f"{BRONZE}: versión {dt.version()}, {bronze.num_rows} filas, {bronze.num_columns} columnas")
print(bronze.schema)

s3://grupo07/bronze/series: versión 223, 190512 filas, 9 columnas
id: int64
date_added: timestamp[us, tz=UTC]
url: string
url_status: string
last_online: string
threat: string
tags: string
urlhaus_link: string
reporter: string


## 2. Exploración: qué problemas tiene el dato crudo

### 2.1 ¿Hay URLs repetidas o sin datos clave?

In [4]:
con.sql("""
    select count(*)                                   as filas,
           count(distinct id)                         as ids_distintos,
           count(*) - count(distinct id)              as filas_repetidas,
           count(*) filter (where id is null)         as sin_id,
           count(*) filter (where date_added is null) as sin_fecha,
           count(*) filter (where tags is null or trim(tags) = '') as sin_tags,
           min(date_added) as primera, max(date_added) as ultima
    from bronze
""").df()

,filas,ids_distintos,filas_repetidas,sin_id,sin_fecha,sin_tags,primera,ultima
0,190512,190512,0,0,0,2044,2018-06-05 22:09:14+00:00,2026-11-19 14:44:23+00:00


→ **Silver deduplica por `id`** y descarta filas sin id o sin fecha, por si una reingesta repite URLs. Las que no tienen etiquetas quedan como `sin_familia` / `desconocido`.

### 2.2 ¿Sirve `threat` para distinguir amenazas?

In [5]:
con.sql("""
    select threat, count(*) as urls, round(100.0 * count(*) / sum(count(*)) over (), 2) as pct
    from bronze group by 1 order by 2 desc
""").df()

,threat,urls,pct
0,malware_download,190510,100.0
1,c2,2,0.0


→ Prácticamente un solo valor, `malware_download` (solo 2 URLs con `c2`): **no distingue nada**. Por eso silver deduce dos dimensiones nuevas de las etiquetas: la **familia de malware** y el **tipo de archivo**.

### 2.3 ¿Qué hay en las etiquetas?

In [6]:
etiquetas = con.sql(r"""
    with t as (select trim(unnest(string_split(regexp_replace(coalesce(tags, ''), '[\[\]''"]', '', 'g'), ','))) as tag
               from bronze)
    select tag, count(*) as urls from t where tag <> '' group by 1 order by 2 desc limit 40
""").df()
etiquetas

,tag,urls
0,github,114061
1,zip,68691
2,SmartLoader,59025
3,zapret,49152
4,SalatStealer,33777
5,elf,23519
6,Mozi,20575
7,mirai,18711
8,LuaJIT-loader,18387
9,ua-wget,12918


Las etiquetas mezclan la familia (`Mozi`, `mirai`, `LuaJIT-loader`) con el tipo de archivo (`elf`, `exe`, `zip`), la arquitectura (`32-bit`, `mips`, `arm`), el lugar donde estaba alojado (`github`, `discord`), el país y etiquetas de quien reporta (`dropped-by-*`, `ua-*`). Y aparecen cosas peores:

In [7]:
con.sql(r"""
    with t as (select trim(unnest(string_split(regexp_replace(coalesce(tags, ''), '[\[\]''"]', '', 'g'), ','))) as tag
               from bronze)
    select case
             when regexp_matches(tag, '^[0-9]{1,3}([.-][0-9]{1,3}){3}([.:-][0-9]{1,5})?$') then 'IP'
             when regexp_matches(lower(tag), '[.-](com|net|org|xyz|top|site|space|to|ru|cn)$') then 'dominio'
             when regexp_matches(lower(tag), '^[0-9a-f]{6,}$') then 'pedazo de hash'
             when lower(tag) in ('cowrie', 'honeypot', 'ssh', 'telnet') then 'honeypot / protocolo'
           end as tipo_de_ruido,
           count(distinct tag) as etiquetas_distintas, count(*) as urls,
           any_value(tag) as ejemplo
    from t
    group by 1 having tipo_de_ruido is not null order by 3 desc
""").df()

,tipo_de_ruido,etiquetas_distintas,urls,ejemplo
0,IP,773,5231,130-12-181-99
1,honeypot / protocolo,4,2581,cowrie
2,dominio,31,392,aura-tcplegit-space
3,pedazo de hash,39,372,54e64e


→ **Silver toma como familia la primera etiqueta que no es ninguna de esas cosas.** Las IPs, los dominios, los honeypots y los hashes los encontramos recién con datos reales, cuando aparecieron como "familias" en el tablero. El test `assert_familia_no_es_tag_generico` evita que vuelvan.

### 2.4 ¿La misma familia aparece escrita distinto?

In [8]:
con.sql(r"""
    with t as (select trim(unnest(string_split(regexp_replace(coalesce(tags, ''), '[\[\]''"]', '', 'g'), ','))) as tag
               from bronze)
    select lower(tag) as familia, list(distinct tag) as grafias, count(*) as urls
    from t where tag <> ''
    group by 1 having count(distinct tag) > 1
    order by 3 desc limit 10
""").df()

,familia,grafias,urls
0,elf,"[elf, ELF]",23522
1,vscode,"[VSCode, vscode]",901
2,stealer,"[stealer, Stealer]",642
3,dropper,"[dropper, Dropper]",617
4,macos,"[macOS, macos]",171
5,amos,"[Amos, amos]",81
6,miner,"[miner, Miner]",23
7,nodejs,"[NodeJS, nodeJS]",21
8,yogi,"[yogi, Yogi]",18
9,react2shell,"[react2shell, React2Shell]",9


→ `Mozi` / `mozi`, `Mirai` / `mirai`, ... **Silver unifica con la grafía más frecuente**, si no una misma familia se partiría en dos en el ranking.

### 2.5 ¿Cuántas URLs hay por mes? (cobertura)

In [9]:
por_mes = con.sql("""
    select strftime(date_added, '%Y-%m') as mes, count(*) as urls
    from bronze group by 1 order by 1
""").df()
print(por_mes.tail(14).to_string(index=False))
print(f"\nMeses con datos: {len(por_mes)} · meses con menos de 1.000 URLs: {(por_mes.urls < 1000).sum()}")

    mes   urls
2025-10   1072
2025-11    148
2025-12     79
2026-01    145
2026-02    168
2026-03    248
2026-04    314
2026-05     75
2026-06   4748
2026-07   4598
2026-08  15663
2026-09  15203
2026-10 122726
2026-11  21291

Meses con datos: 93 · meses con menos de 1.000 URLs: 85


→ **Antes de agosto de 2026 los meses tienen muchas menos URLs:** el export de URLhaus conserva bien solo los últimos uno o dos meses y ya no publica las URLs viejas. Es un hueco del origen, no del pipeline, y está documentado en la governance. El salto de octubre, en cambio, es real: un pico de reportes (ver 2.6).

### 2.6 ¿Quién reporta?

In [10]:
con.sql("""
    select reporter, count(*) as urls, round(100.0 * count(*) / sum(count(*)) over (), 1) as pct,
           min(date_added)::date as desde, max(date_added)::date as hasta
    from bronze group by 1 order by 2 desc limit 8
""").df()

,reporter,urls,pct,desde,hasta
0,wristhulk,96965,50.9,2026-09-29,2026-11-19
1,shmulc,15227,8.0,2026-10-06,2026-10-18
2,adliwahid,15223,8.0,2026-03-20,2026-11-19
3,geenensp,10678,5.6,2022-02-19,2026-11-19
4,cesnet_certs,6704,3.5,2024-08-30,2026-11-18
5,anonymous,6431,3.4,2018-08-02,2026-11-19
6,BlinkzSec,6275,3.3,2024-05-30,2026-11-18
7,abuse_ch,6012,3.2,2018-09-23,2026-11-19


→ **Un solo reportero, `wristhulk`, cargó desde el 29/09/2026 la mitad de todas las URLs**, y ese período es exactamente el pico de octubre. Un pico en el tablero puede ser un reportero muy activo y no más malware: queda como limitación en la governance. Además, `reporter` identifica a una persona o equipo: se queda en silver y no se publica.

## 3. Historial de versiones (*time travel*)

Delta guarda en su log cada operación sobre la tabla, y en principio permite leer cualquier versión anterior.

In [11]:
historia = pd.DataFrame(dt.history(limit=10))
historia["timestamp"] = pd.to_datetime(historia["timestamp"], unit="ms", utc=True)
historia[[c for c in ["version", "timestamp", "operation", "operationParameters"] if c in historia.columns]]

,version,timestamp,operation,operationParameters
0,223,2026-11-19 15:40:44.515000+00:00,VACUUM END,{'status': 'COMPLETED'}
1,222,2026-11-19 15:40:44.489000+00:00,VACUUM START,"{'specifiedRetentionMillis': '0', 'retentionCheckEnabled': 'false', 'default..."
2,221,2026-11-19 15:40:44.433000+00:00,MERGE,"{'mergePredicate': 't.id = s.id', 'notMatchedPredicates': '[{""actionType"":""i..."
3,220,2026-11-19 14:51:44.878000+00:00,WRITE,{'mode': 'Append'}
4,219,2026-11-19 14:17:36.930000+00:00,VACUUM END,{'status': 'COMPLETED'}
5,218,2026-11-19 14:17:36.908000+00:00,VACUUM START,"{'specifiedRetentionMillis': '0', 'retentionCheckEnabled': 'false', 'default..."
6,217,2026-11-19 14:17:36.856000+00:00,MERGE,"{'notMatchedPredicates': '[{""actionType"":""insert""}]', 'matchedPredicates': '..."
7,216,2026-11-19 03:01:41.489000+00:00,VACUUM END,{'status': 'COMPLETED'}
8,215,2026-11-19 03:01:41.462000+00:00,VACUUM START,"{'retentionCheckEnabled': 'false', 'defaultRetentionMillis': '604800000', 's..."
9,214,2026-11-19 03:01:41.385000+00:00,MERGE,"{'notMatchedBySourcePredicates': '[]', 'notMatchedPredicates': '[{""actionTyp..."


**Las versiones viejas de bronze ya no se pueden leer.** La inmediatamente anterior a veces sí: si después de ella no corrió ningún `vacuum` (por ejemplo, una corrida de NB02, que solo agrega), sus archivos siguen siendo parte de la tabla. Pero después de escribir, `extraer_bronze` hace `vacuum` con retención 0, porque dbt lee los `.parquet` directamente y sin `vacuum` leería también archivos que ya no son parte de la tabla. Eso borra los archivos de las versiones anteriores: el log las recuerda, pero sus datos ya no están. Lo sufrimos el 18/11/2026, cuando un reproceso borró URLs y el *time travel* no tenía a dónde volver (ver `docs/INCIDENTE_2026-11-18_backfill.md`).

In [12]:
# La anterior, una de unas horas atrás y la primera
for v in sorted({max(dt.version() - 1, 0), max(dt.version() - 10, 0), 0}, reverse=True):
    try:
        n = DeltaTable(BRONZE, version=v, storage_options=STORAGE_OPTIONS).to_pyarrow_table(columns=["id"]).num_rows
        print(f"Versión {v}: se puede leer ({n} filas)")
    except Exception as e:
        print(f"Versión {v}: no se puede leer ({type(e).__name__}): sus archivos los borró un vacuum posterior")

Versión 222: se puede leer (190512 filas)
Versión 213: no se puede leer (FileNotFoundError): sus archivos los borró un vacuum posterior
Versión 0: no se puede leer (FileNotFoundError): sus archivos los borró un vacuum posterior


Para ver el *time travel* funcionando, una tabla de demostración aparte (no toca bronze): se escribe dos veces y se lee la versión 0.

In [13]:
from deltalake import write_deltalake
import pyarrow as pa

DEMO = f"s3://{BUCKET}/demo/time_travel"
write_deltalake(DEMO, pa.table({"id": [1, 2, 3], "estado": ["online"] * 3}),
                mode="overwrite", storage_options=STORAGE_OPTIONS)
write_deltalake(DEMO, pa.table({"id": [4, 5], "estado": ["offline"] * 2}),
                mode="append", storage_options=STORAGE_OPTIONS)

actual = DeltaTable(DEMO, storage_options=STORAGE_OPTIONS)
inicio = actual.version() - 1
v0 = DeltaTable(DEMO, version=inicio, storage_options=STORAGE_OPTIONS)
print(f"Versión actual ({actual.version()}): {actual.to_pyarrow_table().num_rows} filas")
print(f"Versión {inicio} (antes del append): {v0.to_pyarrow_table().num_rows} filas")

Versión actual (3): 5 filas
Versión 2 (antes del append): 3 filas


## Cierre

DuckDB sobre Delta permite explorar el lake con SQL sin moverlo a una base. Lo que encontramos acá (un solo `threat`, etiquetas con ruido, grafías distintas, huecos de cobertura, reporteros dominantes) es lo que limpia silver y lo que documenta la governance. Pendiente de mejora: si dbt leyera la tabla Delta con `delta_scan` en vez de los `.parquet` sueltos, se podría guardar historia de versiones (sacar el `vacuum` con retención 0) y tener *time travel* real sobre bronze.